# 1. Set-up

In the code bellow packages are imported and the PlantMetwiki dataset is convverted from CSV to a Dataframe.


## Python packages

In [274]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import requests
import match_plantmetwiki as m

## Load in PlantMetWiki data

In [ ]:
# df_plant_met_wiki = pd.read_csv("./data/plantmetwiki_metabolites.csv")
# InChiKey_full = df_plant_met_wiki
# InChiKey_full.head()

,inchikey,name,wikidata,n_pathways
0,COHYTHOBJLSHDF-BUHFOSPRSA-N,indigo,http://www.wikidata.org/entity/Q422662,2
1,ITYHVANGBZMQML-BQUKFSKHSA-L,"CDP-1,2-dipalmitoylglycerol",http://www.wikidata.org/entity/Q27146750,4
2,NOTGFIUVDGNKRI-UUOKFMHZSA-L,5-amino-1-(5-phospho-D-ribosyl)imidazole-4-car...,http://www.wikidata.org/entity/Q27125815,8
3,WACBUPFEGWUGPB-MIUGBVLSSA-N,calycosin 7-O-glucoside,http://www.wikidata.org/entity/Q27159206,2
4,IHPKGUQCSIINRJ-CSKARUKUSA-N,(E)-β-ocimene,http://www.wikidata.org/entity/Q26777012,4


# 2. Inchikeys to Smiles

In the code bellow Inchikeys are converted to smiles ussing the PubChem database. Note some Metabolites have know Smiles, these metabolites will have NaN.


In [ ]:
def key_to_smiles(inchikey):
    """
    """
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/%s/property/IsomericSMILES/TXT" % inchikey
    response = requests.get(url, timeout=30)
    smiles = response.text.strip()
    if smiles.startswith('Status: 404'):
        smiles = None
    return smiles

In [ ]:
# InChiKey_full["smiles"] = InChiKey_full["inchikey"].apply(key_to_smiles)
# InChiKey_full.columns

Index(['inchikey', 'name', 'wikidata', 'n_pathways', 'smiles'], dtype='str')

# 3. Smiles to NPC classifiation
In the code below all metabolites with a smiles are classified using the NPclasiefier website as adapted from the example query provided in the suplementary materials of Kim, H. et al (2021).


Kim, H. W., Wang, M., Leber, C. A., Nothias, L. F., Reher, R., Kang, K. B., van der Hooft, J. J. J., Dorrestein, P. C., Gerwick, W. H., & Cottrell, G. W. (2021). NPClassifier: A Deep Neural Network-Based Structural Classification Tool for Natural Products. Journal of natural products, 84(11), 2795–2807. https://doi.org/10.1021/acs.jnatprod.1c00399

In [281]:
def smiles_to_NPCs(smiles):
    if smiles != None:
            url = f"https://npclassifier.gnps2.org/classify?smiles=%s" %smiles
            response = requests.get(url, timeout = 30)
            if response.status_code != 200:
                             path = None
                             super = None
                             sub = None  
            else: 
                pathway = response.json()["pathway_results"]
                superclass = response.json()["superclass_results"]
                subclass = response.json()["class_results"]

                if len(pathway) == 1:
                    path = pathway[0] 
                else:
                    path = None

                if len(superclass) == 1:
                    super = superclass[0] 
                else:
                    super = None

                if len(subclass) == 1:
                    sub = subclass[0]
                else:
                    sub = None  
    else:
         path = None
         super = None
         sub = None  
    return path, super, sub

def split_ncp_class_in_path(ncp_classification):
    path, super, sub = ncp_classification
    return path

def split_ncp_class_in_superclass(ncp_classification):
    path, super, sub = ncp_classification
    return super

def split_ncp_class_in_class(ncp_classification):
    path, super, sub = ncp_classification
    return sub



In [ ]:

# # Gather NPC clasification
# InChiKey_full["ncp_classification"] = InChiKey_full["smiles"].apply(smiles_to_NPCs)

# #split NPC clasfication into 3 collumns
# InChiKey_full["ncp_pathway"] = InChiKey_full["ncp_classification"].apply(split_ncp_class_in_path)
# InChiKey_full["ncp_superclass"] = InChiKey_full["ncp_classification"].apply(split_ncp_class_in_superclass)
# InChiKey_full["ncp_class"] = InChiKey_full["ncp_classification"].apply(split_ncp_class_in_class)

In [283]:
# write data To CSV file
InChiKey_full.to_csv("All_NPC_classifications.csv", index=True, header=True)

# 4. An overview of NPC clasification Coverage
In the code below An overview of the coverage of NPC classification is shown for the metabolites in Plant met Wiki.

In [284]:
# counting the NaN, these Inchikeys have no smiles linked in Pubchem
no_smiles = InChiKey_full["smiles"].isna().sum()
print(no_smiles)


85


In [285]:
Metabolites_keys= InChiKey_full["inchikey"].count()
Metabolites_smiles = InChiKey_full["smiles"].count()
Metabolites_pathway =InChiKey_full["ncp_pathway"].count()
Metabolites_superclass =InChiKey_full["ncp_superclass"].count()
Metabolites_class = InChiKey_full["ncp_class"].count()

print(f"Number of metabolites checked : {Metabolites_keys} \nNumber of metabolites who have a smiles: {Metabolites_smiles}\nNumber of metabolites who have A NPC pathway classification: {Metabolites_pathway} \nNumber of metabolites who have A NPC superclass classification: {Metabolites_superclass} \nNumber of metabolites who have A NPC class classification: {Metabolites_class}")

Number of metabolites checked : 2974 
Number of metabolites who have a smiles: 2889
Number of metabolites who have A NPC pathway classification: 2324 
Number of metabolites who have A NPC superclass classification: 2178 
Number of metabolites who have A NPC class classification: 1953


# 5. Run-Time.

The WikiPathway set contains circa 3000 metabolites as of 12 november 2026.

As this Notebook uses a query to collect a metabolites smiles and NPC classification. We have made an over view of the average runtimes based on the amount of metabolite inputed. For both searches the runtime has been capped at 30 seconds, if it takes longer then 30 seconds to find something the assumption is made that there is nothing to find, thus the smiles or NPC classification is set to None. Worse case scenario, al seaches go the full time for a combined 60 seconds. In that scenario a 3000 metabolite search would take 50 hours. 

For a subset of 10 metabolites it took 40 seconds to gather the smiles and another 60 seconds to run the NPC. (worse case: 10 minutes) 
For a subset of 100 metabolites it took 40 seconds to gather the smiles and another 60 seconds to run the NPC. (worse case: 1 hour and 40 minutes)
For a subset of 1000 Metabolites it  took 6 minutes and 6 seconds to collect the smiles and 9 minutes and 11 seconds to collect the NPC. (worse case: 17 hours)
For a subset of 1000 Metabolites It takes 17 minutes to collect teh smiles and 28 minutes to collect the NPC. (worse case: 50 hours)